# Run Aegis QEC for free in Google Colab or Jupyter

This notebook uses a **temporary Python session**. Run the cells from top to bottom. The first code cell installs the released [Aegis QEC](https://pypi.org/project/aegis-qec/) package and its scientific dependencies. In Colab, the CPU is enough for these small examples; a GPU is not needed. Free Colab resources and session time are not guaranteed.

These experiments simulate quantum surface-code errors and decode them. They do **not** represent quantum-hardware measurements or demonstrate a physical threshold. For the complete source, [research guide](https://github.com/hamidbahri92/Aegis-QEC/blob/main/docs/RESEARCH_GUIDE.md), and [community](https://github.com/hamidbahri92/Aegis-QEC/discussions/53), visit the Aegis QEC GitHub repository.


In [ ]:
%pip install -q "aegis-qec[full]"
from importlib.metadata import version
print('Aegis QEC', version('aegis-qec'))
print('Stim', version('stim'))
print('PyMatching', version('pymatching'))


In [ ]:
from aegis_qec.explain import explain_surface_code_shot
record = explain_surface_code_shot(distance=5, physical_error_rate=0.006, basis='x', seed=1234)
print('Fired detectors:', record['fired_detector_count'])
print('Matched detection events:', record['matched_detection_events'])
print('Logical prediction correct:', not record['logical_failure'])
print('Circuit SHA256:', record['provenance']['circuit_sha256'])


## Compare additional experimental decoders

The remaining example illustrates Aegis's additional decoder interfaces on a small deterministic syndrome. Some are experimental research paths and must be independently validated before scientific performance claims.


# Aegis Educational Demo
This notebook shows how to build a layout, run decoders (MWPM, UF‑E, MWPM2, MWPM‑Corr), and compare results.


In [ ]:
from a3d import AegisConfig, DecoderRuntime, RotatedSurfaceLayout

cfg = AegisConfig(distance=3, rounds=3, decoder_type='mwpm')
layout = RotatedSurfaceLayout(cfg.distance)
rt = DecoderRuntime(cfg, layout)
nX = len(rt.builder.node_order('X'))
sX = [0]*nX
nZ = len(rt.builder.node_order('Z'))
sZ = [0]*nZ
resX, resZ = rt.decode_from_syndromes_uniform(sX, sZ)
print('MWPM avg_cost:', resX.avg_cost, resZ.avg_cost)


## Try the pipelined and correlation-aware MWPM


In [ ]:
cfg.decoder_type = 'mwpm2'
rt = DecoderRuntime(cfg, layout)
resX2, resZ2 = rt.decode_from_syndromes_uniform(sX, sZ)
print('MWPM2 avg_cost:', resX2.avg_cost, resZ2.avg_cost)
cfg.decoder_type = 'mwpm_corr'
rt = DecoderRuntime(cfg, layout)
resX3, resZ3 = rt.decode_from_syndromes_uniform(sX, sZ)
print('MWPM-Corr avg_cost:', resX3.avg_cost, resZ3.avg_cost)


## UF‑E with erasure peeling


In [ ]:
cfg.decoder_type = 'uf'
rt = DecoderRuntime(cfg, layout)
resXu, resZu = rt.decode_from_syndromes_uniform(sX, sZ, perase_time=0.8)
print('UF-E avg_cost:', resXu.avg_cost, resZu.avg_cost)
